In [2]:
import os
import polars as pl
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from datetime import datetime, timedelta
import time
from sqlalchemy import create_engine
import pymssql
from typing import List, Dict

import warnings
warnings.filterwarnings('ignore')

In [3]:
T,N = 3400, 5422
start_dt = '2008-01-01'     
end_dt = '2025-12-31'

JY_CONFIG = {
    "server": '10.10.0.102',
    "user": 'jydbReader',
    "password": 'jy@9043!Reader',
    "database": 'jydb',
    "charset": 'cp936'
}
JY_CONN = pymssql.connect(**JY_CONFIG)

ROOT = '/data/xujiayi/end2end/d_field/'

#### Basics

In [5]:
sql_axis = f'''select
                C.SecuCode as "tick",
                A.TradingDay as "date"
            from QT_StockPerformance A
            left join SecuMain C
            on A.InnerCode = C.InnerCode
            where A.TradingDay between '{start_dt}' and '{end_dt}'
                and C.SecuMarket in (83,90)
                and C.SecuCategory=1
            union all
            select
                C.SecuCode as "tick",
                B.TradingDay as "date"
            from LC_STIBPerformance B
            left join SecuMain C
            on B.InnerCode = C.InnerCode
            where B.TradingDay between '{start_dt}' and '{end_dt}'
                and C.SecuMarket in (83,90)
                and C.SecuCategory=1
        '''
axis = pl.read_database(sql_axis, JY_CONN).sort('tick','date')

dates = axis['date'].unique().sort().to_numpy()
#np.save('/data/xujiayi/end2end/axis/dates.npy', dts, allow_pickle=True)

ticks = axis['tick'].unique().sort().to_numpy()
#np.save('/data/xujiayi/end2end/axis/ticks.npy', ticks, allow_pickle=True)

In [4]:
sql_fuquan = f'''select
                   A.ExDiviDate as "date",
                   B.SecuCode as "tick",
                   A.AdjustingFactor as "fq_factor"
                from QT_StockAdjustFactor A
                left join SecuMain B
                on A.InnerCode = B.InnerCode
                union all
                select
                   C.ExDiviDate as "date",
                   B.SecuCode as "tick",
                   C.RatioAdjustingFactor as "fq_factor"
                from LC_STIBAdjustingFactor C
                left join SecuMain B
                on C.InnerCode = B.InnerCode
             '''
fuquan = pl.read_database(sql_fuquan, JY_CONN).sort('tick','date')
fuquan = axis.join(
    fuquan,
    how='left',
    on=['tick','date']
).with_columns(
    pl.col('fq_factor').forward_fill().over('tick')
)
fuquan = fuquan.pivot(index='date', columns='tick', values='fq_factor').fill_null(1)
fuquan = fuquan.drop('date').to_numpy().astype(float)

fuquan

array([[  1.      ,   1.      ,   1.      , ...,   1.      ,   1.      ,
          1.      ],
       [  1.      ,   1.      ,   1.      , ...,   1.      ,   1.      ,
          1.      ],
       [  1.      ,   1.      ,   1.      , ...,   1.      ,   1.      ,
          1.      ],
       ...,
       [154.496923, 116.819198,   1.      , ...,   1.      ,   1.093522,
          1.      ],
       [154.496923, 116.819198,   1.      , ...,   1.      ,   1.093522,
          1.      ],
       [154.496923, 116.819198,   1.      , ...,   1.      ,   1.093522,
          1.      ]], shape=(4376, 5436))

In [5]:
fuquan.tofile('/data/xujiayi/end2end/d_field/fuquan.bin')

In [ ]:
dates = np.load('/data/xujiayi/end2end/axis/dates.npy', allow_pickle=True)
ticks = np.load('/data/xujiayi/end2end/axis/ticks.npy', allow_pickle=True)
fuquan = np.memmap(f'/data/xujiayi/end2end/d_field/fuquan.bin',shape=(len(dates),len(ticks)), dtype=float, mode='r')
DAILY_FEATURES = ['open','high','low','close','volume']
for feat in DAILY_FEATURES:
    arr = np.memmap(f'/data/xujiayi/end2end/d_field/{feat}.bin',shape=(len(dates),len(ticks)), dtype=float, mode='r')
    arr_adj  = arr*fuquan
    print(arr_adj)
    arr_adj.astype(float).tofile(f'/data/xujiayi/end2end/d_field/{feat}_adj.bin')

[[  38.5          29.02          9.86       ...           nan
            nan           nan]
 [  38.           29.1          10.4        ...           nan
            nan           nan]
 [  37.4          28.6          10.8        ...           nan
            nan           nan]
 ...
 [1782.89449142  554.8911905    10.9        ...           nan
    37.13600712  123.        ]
 [1781.34952219  551.38661456   10.84       ...  265.6
    36.37054172  124.45      ]
 [1773.62467604  544.37746268   11.3        ...  240.
    36.4689587   125.99      ]]
[[  38.74         29.51         10.46       ...           nan
            nan           nan]
 [  38.08         29.1          10.68       ...           nan
            nan           nan]
 [  38.7          29.6          10.99       ...           nan
            nan           nan]
 ...
 [1795.25424526  559.56395842   11.2        ...           nan
    37.13600712  125.65      ]
 [1785.98442988  552.55480654   11.41       ...  276.82
    36.73140398  1

In [ ]:
fields = {
    'ClosePrice' : "close",
    'HighPrice' : "high",
    'LowPrice' : "low",
    'OpenPrice' : "open",
    "TurnoverVolume": 'volume',
    'TurnoverValue' : "amount",
    'TurnoverRate' : "turnover",
}

for k, v in tqdm(fields.items()):
    sql_field = f'''select
                        C.SecuCode as "tick",
                        A.TradingDay as "date",
                        A.{k} as "{v}"
                    from QT_StockPerformance A
                    left join SecuMain C
                    on A.InnerCode = C.InnerCode
                    where A.TradingDay between '{start_dt}' and '{end_dt}'
                        and C.SecuMarket in (83,90)
                        and C.SecuCategory=1
                    union all
                    select
                        C.SecuCode as "tick",
                        B.TradingDay as "date",
                        B.{k} as "{v}"
                    from LC_STIBPerformance B
                    left join SecuMain C
                    on B.InnerCode = C.InnerCode
                    where B.TradingDay between '{start_dt}' and '{end_dt}'
                        and C.SecuMarket in (83,90)
                        and C.SecuCategory=1
                 '''
    d_field = pl.read_database(sql_field, JY_CONN).sort('tick','date')
    d_field = d_field.pivot(index='date', columns='tick', values=v).drop('date').to_numpy().astype(float)
    if v=='amount' or v=='turnover':
        d_field.tofile(f"/data/xujiayi/end2end/d_field/{v}.bin")
    else:
        d_field.tofile(f"/data/xujiayi/end2end/d_field/{v}.bin")
        d_field_adj = d_field*fuquan
        d_field_adj.tofile(f"/data/xujiayi/end2end/d_field/{v}_adj.bin")

In [ ]:
sql_pct = f'''select
                        C.SecuCode as "tick",
                        A.TradingDay as "date",
                        A.ChangePCT as "pct"
                    from QT_StockPerformance A
                    left join SecuMain C
                    on A.InnerCode = C.InnerCode
                    where A.TradingDay between '2007-01-01' and '2026-03-01'
                        and C.SecuMarket in (83,90)
                        and C.SecuCategory=1
                    union all
                    select
                        C.SecuCode as "tick",
                        B.TradingDay as "date",
                        B.ChangePCT as "pct"
                    from LC_STIBPerformance B
                    left join SecuMain C
                    on B.InnerCode = C.InnerCode
                    where B.TradingDay between '2007-01-01' and '2026-03-01'
                        and C.SecuMarket in (83,90)
                        and C.SecuCategory=1
            '''
pct = pl.read_database(sql_pct, JY_CONN).sort('tick','date')
pct = pct.pivot(index='date', columns='tick', values='pct').to_pandas().set_index('date')
pct = pct.shift(-1)
pct = pct/100
pct = pct.reindex(index=dates, columns=ticks)
pct.values.astype(float).tofile('/data/xujiayi/end2end/label/Y.1D.bin')

In [ ]:
#pct = pct.values.astype(float)
industry_mask = np.memmap('/data/xujiayi/end2end/mask/industry.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
logmv = np.memmap('/data/xujiayi/end2end/d_field/logmv.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')

from ..utils import Calculator, Processor
for i in [1,3,5,10]:
    y = Calculator.rolling_retprod(pct,i,True)
    y.astype(float).tofile(f'/data/xujiayi/end2end/label/Y.{i}D.bin')
    y_yeo = Processor.yeojohnson(y)
    y_yeo.astype(float).tofile(f'/data/xujiayi/end2end/label/Yyeo.{i}D.bin')
    y_z = Processor.cross_standardize(y)
    y_z.astype(float).tofile(f'/data/xujiayi/end2end/label/Yz.{i}D.bin')
    y_dm = Processor.indmv_neutral_longshort(y, industry_mask, logmv)
    y_dm.astype(float).tofile(f'/data/xujiayi/end2end/label/Ydm.{i}D.bin')
    y_r = Processor.rank_transform(y)
    y_r.astype(float).tofile(f'/data/xujiayi/end2end/label/Yr.{i}D.bin')
    y_ls = Processor.winsorize_linearsmooth(y)
    y_ls.astype(float).tofile(f'/data/xujiayi/end2end/label/Yls.{i}D.bin')

In [18]:
sql_ceil_floor = f'''
    select
        A.TradingDay as 'date',
        B.SecuCode as 'tick',
        A.PriceCeiling as "ceil",
        A.PriceFloor as "floor"
    from QT_PriceLimit A
    left join SecuMain B on A.InnerCode = B.InnerCode
    where A.TradingDay between '{start_dt}' and '{end_dt}'
        and B.SecuMarket in (83,90)
        and B.SecuCategory=1
    '''
hitprice_df = pl.read_database(sql_ceil_floor, JY_CONN).sort('tick','date')
ceil = hitprice_df.pivot(index='date', columns='tick', values='ceil').to_pandas().set_index('date').reindex(columns=ticks)
floor = hitprice_df.pivot(index='date', columns='tick', values='floor').to_pandas().set_index('date').reindex(columns=ticks)


In [20]:
fuquan = np.memmap('/data/xujiayi/end2end/d_field/fuquan.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')

ceil.values.astype(float).tofile(f"/data/xujiayi/end2end/d_field/ceil.bin")
ceil_adj = ceil.values.astype(float)*fuquan
ceil_adj.astype(float).tofile(f"/data/xujiayi/end2end/d_field/ceil_adj.bin")

floor.values.astype(float).tofile(f"/data/xujiayi/end2end/d_field/floor.bin")
floor_adj = floor.values.astype(float)*fuquan
floor_adj.astype(float).tofile(f"/data/xujiayi/end2end/d_field/floor_adj.bin")

In [ ]:
def get_next_trading_day(date) -> str:
    date_dt = datetime.strptime(date, "%Y-%m-%d")
    # 读取全市场交易日历（只需查一次，可优化为全局缓存）
    sql_trading_days = """
    SELECT DISTINCT TradingDay as date
    FROM QT_StockPerformance
    WHERE TradingDay >= DATEADD(day, 1, '{date}')
    ORDER BY TradingDay ASC
    """
    trading_days_df = pl.read_database(sql_trading_days.format(date=date), JY_CONN)
    if trading_days_df.height == 0:
        return date  # 无后续交易日，返回原日期
    next_day = trading_days_df["date"][0]
    return datetime.strftime(next_day, "%Y-%m-%d")


def is_st(date, listed_ticks) -> bool:
    """是否ST股"""
    listed_ticks_str = ",".join([f"'{t}'" for t in listed_ticks])

    sql_isst = f'''
    select distinct B.SecuCode as tick, A.SpecialTradeType as "type"
    from LC_SpecialTrade A
    left join SecuMain B on A.InnerCode = B.InnerCode
    where A.InfoPublDate = '{date}' and B.SecuCode in ({listed_ticks_str})
    union all
    select distinct B.SecuCode as tick, C.ChangeType as "type"
    from LC_STIBSecuChange C
    left join SecuMain B on C.InnerCode = B.InnerCode
    where C.InfoPublDate = '{date}' and B.SecuCode in ({listed_ticks_str})
    '''
    st_df = pl.read_database(sql_isst, JY_CONN)
    # 筛选有效ST记录，提取ST股票代码集合
    st_tick_set = set(st_df.filter(pl.col("type").is_not_null())["tick"].to_list())
    return np.array([tick in st_tick_set for tick in listed_ticks])


def is_suspend(date, listed_ticks) -> bool:
    """是否停牌"""
    listed_ticks_str = ",".join([f"'{t}'" for t in listed_ticks])

    sql_suspend = f'''
    select distinct C.SecuCode as tick, A.Ifsuspend as "is_suspend"
    from QT_StockPerformance A
    left join SecuMain C on A.InnerCode = C.InnerCode
    where A.TradingDay = '{date}' and C.SecuCode in ({listed_ticks_str}) and C.SecuMarket in (83,90) and C.SecuCategory=1
    union all
    select distinct C.SecuCode as tick, B.Ifsuspend as "is_suspend"
    from LC_STIBPerformance B
    left join SecuMain C on B.InnerCode = C.InnerCode
    where B.TradingDay = '{date}' and C.SecuCode in ({listed_ticks_str}) and C.SecuMarket in (83,90) and C.SecuCategory=1
    '''
    suspend_df = pl.read_database(sql_suspend, JY_CONN)
    if suspend_df.height == 0:
        # 无数据默认停牌
        return [True for _ in listed_ticks]
    # 构建{股票代码: 是否停牌}字典
    suspend_dict = dict(zip(suspend_df["tick"], suspend_df["is_suspend"].fill_null(1).to_list()))
    # 按输入顺序返回，未查到的股票默认停牌
    return [suspend_dict.get(tick, 1) == 1 for tick in listed_ticks]


def is_new(date, listed_ticks) -> bool:
    """是否上市不满240天"""
    listed_ticks_str = ",".join([f"'{t}'" for t in listed_ticks])

    sql_listdate = f'''
    select SecuCode as tick, COALESCE(ListedDate, '1900-01-01') as "list_date"
    from SecuMain
    where SecuCode in ({listed_ticks_str}) and SecuMarket in (83,90) and SecuCategory=1
    '''
    list_df = pl.read_database(sql_listdate, JY_CONN)
    list_dict = dict(zip(list_df["tick"], list_df["list_date"].to_list())) if list_df.height > 0 else {}
    date_dt = datetime.strptime(date, "%Y-%m-%d")
    return [
        True if tick not in list_dict
        else list_dict[tick] == '1900-01-01'
             or (date_dt - list_dict[tick]).days < 242
        for tick in listed_ticks
    ]


def is_hit_ceil_floor(date, listed_ticks) -> (bool, bool):
    """是否涨停/跌停"""
    listed_ticks_str = ",".join([f"'{t}'" for t in listed_ticks])

    sql_hit = f'''
    select distinct B.SecuCode as tick, A.StockBoard as "hit_ceil", A.LimitBoard as "hit_floor"
    from QT_PerformanceData A
    left join SecuMain B on A.InnerCode = B.InnerCode
    where A.TradingDay = '{date}' and B.SecuCode in ({listed_ticks_str}) and B.SecuMarket in (83,90) and B.SecuCategory=1
    union all
    select distinct B.SecuCode as tick, C.StockBoard as "hit_ceil", C.LimitBoard as "hit_floor"
    from LC_STIBPerformanceData C
    left join SecuMain B on C.InnerCode = B.InnerCode
    where C.TradingDay = '{date}' and B.SecuCode in ({listed_ticks_str}) and B.SecuMarket in (83,90) and B.SecuCategory=1
    '''
    hit_df = pl.read_database(sql_hit, JY_CONN)
    # 无数据直接返回全False数组
    if hit_df.height == 0:
        return [False] * len(listed_ticks), [False] * len(listed_ticks)

    hit_df = hit_df.with_columns([
        pl.col("hit_ceil").fill_null(0).cast(pl.Int8).eq(1).alias("is_ceil"),
        pl.col("hit_floor").fill_null(0).cast(pl.Int8).eq(1).alias("is_floor")
    ])

    hit_dict = hit_df.select(["tick", "is_ceil", "is_floor"]).to_dict(as_series=False)
    hit_map = dict(zip(hit_dict["tick"], zip(hit_dict["is_ceil"], hit_dict["is_floor"])))

    hit_ceil_list = [hit_map.get(tick, (False, False))[0] for tick in listed_ticks]
    hit_floor_list = [hit_map.get(tick, (False, False))[1] for tick in listed_ticks]
    return hit_ceil_list, hit_floor_list


def is_non_trade(date, listed_ticks) -> bool:
    """是否无成交"""
    listed_ticks_str = ",".join([f"'{t}'" for t in listed_ticks])

    sql_nontrade = f'''
    select distinct B.SecuCode as tick, A.TurnoverVolume as "volume"
    from QT_Performance A
    left join SecuMain B on A.InnerCode = B.InnerCode
    where A.TradingDay = '{date}' and B.SecuCode in ({listed_ticks_str}) and B.SecuMarket in (83,90) and B.SecuCategory=1
    union all
    select distinct B.SecuCode as tick, C.TurnoverVolume as "volume"
    from LC_STIBPerformance C
    left join SecuMain B on C.InnerCode = B.InnerCode
    where C.TradingDay = '{date}' and B.SecuCode in ({listed_ticks_str}) and B.SecuMarket in (83,90) and B.SecuCategory=1
    '''
    trade_df = pl.read_database(sql_nontrade, JY_CONN)
    if trade_df.height == 0:
        # 无数据默认无成交
        return [True for _ in listed_ticks]
    # 构建{股票代码: 成交量}字典
    volume_dict = dict(zip(trade_df["tick"], trade_df["volume"].fill_null(0).to_list()))
    # 按输入顺序返回，未查到的股票默认无成交
    return [volume_dict.get(tick, 0) == 0 for tick in listed_ticks]


def is_tradable(date, listed_ticks):
    st = np.array(is_st(date, listed_ticks))
    new = np.array(is_new(date, listed_ticks))
    suspend = np.array(is_suspend(date, listed_ticks))
    non_trade_today = np.array(is_non_trade(date, listed_ticks))
    hit_ceil_today, hit_floor_today = is_hit_ceil_floor(date, listed_ticks)
    hit_ceil_today = np.array(hit_ceil_today)
    hit_floor_today = np.array(hit_floor_today)

    next_date = get_next_trading_day(date)
    non_trade_next = np.array(is_non_trade(next_date, listed_ticks))
    hit_ceil_next, hit_floor_next = is_hit_ceil_floor(next_date, listed_ticks)
    hit_ceil_next = np.array(hit_ceil_next)
    hit_floor_next = np.array(hit_floor_next)

    tradable = ~(
            st | new | suspend |
            non_trade_today | hit_ceil_today | hit_floor_today |
            non_trade_next | hit_ceil_next | hit_floor_next
    )

    return tradable


tradables = []
dates = pd.to_datetime(dates).strftime('%Y-%m-%d')
for date in tqdm(dates):
    tradable = is_tradable(date,ticks)
    tradables.append(tradable)
tradables = np.stack(tradables)
print(tradables)
tradables.astype(bool).tofile(f'/data/xujiayi/end2end/mask/tradable.bin')

In [7]:
tradable = np.memmap('/data/xujiayi/end2end/mask/tradable.bin',shape=(len(dates),len(ticks)), mode='r', dtype=bool)
tradable

memmap([[ True,  True,  True, ..., False, False, False],
        [ True,  True,  True, ..., False, False, False],
        [ True,  True,  True, ..., False, False, False],
        ...,
        [ True,  True,  True, ..., False,  True,  True],
        [ True,  True,  True, ..., False,  True,  True],
        [ True,  True,  True, ..., False,  True,  True]],
       shape=(4376, 5436))

#### Industry_Mask

In [52]:
sql = f'''
select
   A.InfoPublDate   as 'start_date',
   A.CancelDate     as 'end_date',
   B.SecuCode       as 'tick',
   B.CompanyCode    as 'code',
   A.FirstIndustryName as 'industry'
from DZ_ExgIndustry A
left join SecuMain B
on A.CompanyCode = B.CompanyCode
where A.Standard=38
   and B.SecuMarket in (83,90)
   and B.SecuCategory=1

union all 

select
   C.InfoPublDate   as 'start_date',
   C.CancelDate     as 'end_date',
   B.SecuCode       as 'tick',
   B.CompanyCode    as 'code',
   C.FirstIndustryName as 'industry'
from LC_STIBExgIndustry C
left join SecuMain B
on C.CompanyCode = B.CompanyCode
where C.Standard=38
   and B.SecuMarket in (83,90)
   and B.SecuCategory=1
'''

ind = pl.read_database(sql, JY_CONN).sort('tick')
ind

start_date,end_date,tick,code,industry
datetime[μs],datetime[μs],str,i64,str
1991-04-03 00:00:00,2014-01-01 00:00:00,"""000001""",3,"""金融服务"""
2014-01-01 00:00:00,2021-07-30 00:00:00,"""000001""",3,"""银行"""
2021-07-30 00:00:00,null,"""000001""",3,"""银行"""
1991-01-29 00:00:00,2021-07-30 00:00:00,"""000002""",6,"""房地产"""
2021-07-30 00:00:00,null,"""000002""",6,"""房地产"""
…,…,…,…,…
2026-05-29 00:00:00,null,"""X26031""",19402327,"""纺织服饰"""
2026-05-29 00:00:00,null,"""X26032""",623937,"""电子"""
2026-06-02 00:00:00,null,"""X26033""",629047,"""交通运输"""


In [53]:
ticks = np.load('/data/xujiayi/end2end/axis/ticks.npy', allow_pickle=True)
dts = np.load('/data/xujiayi/end2end/axis/dates.npy', allow_pickle=True)

df_dates = pl.DataFrame({"date": dts}).sort("date")
df_ticks = pl.DataFrame({"tick": ticks}).sort("tick")
df_base = df_dates.join(df_ticks, how="cross")

In [54]:
ind_processed = (
    ind
    .filter(pl.col("tick").is_in(ticks))
    .with_columns([
        pl.col("start_date").cast(pl.Datetime),
        pl.col("end_date").cast(pl.Datetime).fill_null(pl.datetime(2099, 12, 31))
    ])
    .sort(["tick", "start_date"])
)
ind_processed = (
    df_base
    .join_asof(
        ind_processed,
        left_on="date",
        right_on="start_date",
        by="tick",
        strategy="backward"
    )
    .filter(pl.col("date") < pl.col("end_date"))
    .select(["date", "tick", "industry"])
)
ind_processed

date,tick,industry
datetime[μs],str,str
2008-01-02 00:00:00,"""000001""","""金融服务"""
2008-01-02 00:00:00,"""000002""","""房地产"""
2008-01-02 00:00:00,"""000004""","""房地产"""
2008-01-02 00:00:00,"""000005""","""房地产"""
2008-01-02 00:00:00,"""000006""","""房地产"""
…,…,…
2025-12-31 00:00:00,"""688805""","""医药生物"""
2025-12-31 00:00:00,"""688807""","""电子"""
2025-12-31 00:00:00,"""688809""","""电子"""


In [55]:
update_map = {
    "黑色金属": "钢铁",
    "建筑建材": "建筑材料",
    "化工": "基础化工",
    "交运设备": "汽车",
    "信息服务": "传媒",
    "信息设备": "电子",
    "商业贸易": "商贸零售",
    "纺织服装": "纺织服饰",
    "休闲服务": "社会服务",
    "餐饮旅游": "社会服务",
    "电气设备": "电力设备",
}
ind_processed = ind_processed.with_columns(pl.col('industry').replace(update_map).alias("industry"))
ind_processed

date,tick,industry
datetime[μs],str,str
2008-01-02 00:00:00,"""000001""","""金融服务"""
2008-01-02 00:00:00,"""000002""","""房地产"""
2008-01-02 00:00:00,"""000004""","""房地产"""
2008-01-02 00:00:00,"""000005""","""房地产"""
2008-01-02 00:00:00,"""000006""","""房地产"""
…,…,…
2025-12-31 00:00:00,"""688805""","""医药生物"""
2025-12-31 00:00:00,"""688807""","""电子"""
2025-12-31 00:00:00,"""688809""","""电子"""


In [56]:
bank_tick = ind_processed.filter(pl.col('industry')=='银行')['tick'].unique()
financial_tick = ind_processed.filter(pl.col('industry')=='非银金融')['tick'].unique()
coal_tick = ind_processed.filter(pl.col('industry')=='煤炭')['tick'].unique()
petrochemicals_tick = ind_processed.filter(pl.col('industry')=='石油石化')['tick'].unique()

ind_final = (
    ind_processed
    .with_columns([
        # 1. 处理"金融服务"的拆分
        pl.when(pl.col("industry") == "金融服务")
        .then(
            pl.when(pl.col("tick").is_in(bank_tick)).then(pl.lit("银行"))
            .when(pl.col("tick").is_in(financial_tick)).then(pl.lit("非银金融"))
            .otherwise(pl.lit("非银金融"))
        )
        # 2. 处理"采掘"的拆分
        .when(pl.col("industry") == "采掘")
        .then(
            pl.when(pl.col("tick").is_in(coal_tick)).then(pl.lit("煤炭"))
            .when(pl.col("tick").is_in(petrochemicals_tick)).then(pl.lit("石油石化"))
            .otherwise(pl.lit("煤炭"))
        )
        # 3. 其他行业保持不变
        .otherwise(pl.col("industry"))
        .alias("industry")
    ])
)
ind_final

date,tick,industry
datetime[μs],str,str
2008-01-02 00:00:00,"""000001""","""银行"""
2008-01-02 00:00:00,"""000002""","""房地产"""
2008-01-02 00:00:00,"""000004""","""房地产"""
2008-01-02 00:00:00,"""000005""","""房地产"""
2008-01-02 00:00:00,"""000006""","""房地产"""
…,…,…
2025-12-31 00:00:00,"""688805""","""医药生物"""
2025-12-31 00:00:00,"""688807""","""电子"""
2025-12-31 00:00:00,"""688809""","""电子"""


In [57]:
# 1.1 行业→行业值映射
industry_to_id = {
    "商贸零售": 0, "轻工制造": 1, "汽车": 2, "美容护理": 3, "房地产": 4,
    "国防军工": 5, "通信": 6, "煤炭": 7, "交通运输": 8, "公用事业": 9,
    "机械设备": 10, "电力设备": 11, "环保": 12, "食品饮料": 13, "计算机": 14,
    "纺织服饰": 15, "家用电器": 16, "医药生物": 17, "钢铁": 18, "社会服务": 19,
    "有色金属": 20, "非银金融": 21, "综合": 22, "建筑装饰": 23, "农林牧渔": 24,
    "银行": 25, "传媒": 26, "基础化工": 27, "建筑材料": 28, "石油石化": 29, "电子": 30
}

# 1.2 行业→板块映射
industry_to_sector = {
    "商贸零售": "消费", "轻工制造": "消费", "汽车": "制造", "美容护理": "消费", "房地产": "金融地产",
    "国防军工": "制造", "通信": "科技", "煤炭": "周期", "交通运输": "周期", "公用事业": "周期",
    "机械设备": "制造", "电力设备": "制造", "环保": "制造", "食品饮料": "消费", "计算机": "科技",
    "纺织服饰": "消费", "家用电器": "消费", "医药生物": "消费", "钢铁": "周期", "社会服务": "消费",
    "有色金属": "周期", "非银金融": "金融地产", "综合": "制造", "建筑装饰": "周期", "农林牧渔": "周期",
    "银行": "金融地产", "传媒": "科技", "基础化工": "周期", "建筑材料": "周期", "石油石化": "周期", "电子": "科技"
}

# 1.3 板块→板块ID映射
sector_to_id = {
    "消费": 0,
    "制造": 1,
    "金融地产": 2,
    "科技": 3,
    "周期": 4
}

In [58]:
ind_sector_id = (
    ind_final
    .with_columns([
        pl.col("industry").replace(industry_to_id).alias("industry_id"),
        pl.col("industry").replace(industry_to_sector).alias("sector")
    ])
    .with_columns([
        pl.col("sector").replace(sector_to_id).alias("sector_id")
    ])
)

df_industry = (
    ind_sector_id
    .pivot(index="date", columns="tick", values="industry_id")
    .sort("date")
    .with_columns([
        pl.all().exclude("date").cast(pl.Float64).fill_null(np.nan)
    ])
)

df_sector = (
    ind_sector_id
    .pivot(index="date", columns="tick", values="sector_id")
    .sort("date")
    .with_columns([
        pl.all().exclude("date").cast(pl.Float64).fill_null(np.nan)
    ])
)

In [60]:
df_industry.to_pandas().set_index('date').values.astype(float).tofile('/data/xujiayi/end2end/mask/industry.bin')

In [61]:
df_sector.to_pandas().set_index('date').values.astype(float).tofile('/data/xujiayi/end2end/mask/sector.bin')

#### IndCapGraph_Model

In [ ]:
sql1 = f'''
SELECT
    A.TradingDay            AS date,
    B.SecuCode              AS tick,
    A.PE                    AS pe,
    A.PB                    AS pb, 
    A.DebtToAssetValue1     AS debt_ratio                 
FROM (
    SELECT TradingDay, InnerCode, PE, PB, DebtToAssetValue1 FROM DZ_DIndicesForValuation
    UNION ALL
    SELECT TradingDay, InnerCode, PE, PB, DebtToAssetValue1 FROM LC_DIndicesForValuation
) A
LEFT JOIN SecuMain B 
ON A.InnerCode = B.InnerCode
WHERE B.SecuMarket IN (83, 90)
  AND B.SecuCategory = 1
  AND A.tradingday between '{start_dt}' and '{end_dt}'
ORDER BY date, tick
'''
df1 = pl.read_database(sql1, JY_CONN).unique()
df1

date,tick,pe,pb,debt_ratio
datetime[μs],str,"decimal[38,4]","decimal[38,4]","decimal[38,4]"
2011-06-05 00:00:00,"""002496""",41.1387,1.9863,0.0785
2023-01-01 00:00:00,"""300329""",-16.1029,1.8933,0.2179
2018-12-19 00:00:00,"""601858""",17.3575,2.1359,0.1474
2024-11-15 00:00:00,"""600241""",60.8685,1.8271,0.0628
2024-04-13 00:00:00,"""300235""",70.7572,3.1851,0.0067
…,…,…,…,…
2017-08-11 00:00:00,"""002140""",58.9409,2.5801,0.4238
2018-12-29 00:00:00,"""600750""",15.7849,2.2690,0.0571
2023-03-01 00:00:00,"""300107""",58.4180,2.1853,0.0348


In [ ]:
# 标记重复行（包含所有副本）
df1 = df1.with_columns(
    pl.struct(["date", "tick"]).is_duplicated().alias("is_dup")
)

# 查看所有重复行
dup_rows = df1.filter(pl.col("is_dup"))
print("🔥 重复行数量：", len(dup_rows))
print(dup_rows)

🔥 重复行数量： 0
shape: (0, 6)
┌──────────────┬──────┬───────────────┬───────────────┬───────────────┬────────┐
│ date         ┆ tick ┆ pe            ┆ pb            ┆ debt_ratio    ┆ is_dup │
│ ---          ┆ ---  ┆ ---           ┆ ---           ┆ ---           ┆ ---    │
│ datetime[μs] ┆ str  ┆ decimal[38,4] ┆ decimal[38,4] ┆ decimal[38,4] ┆ bool   │
╞══════════════╪══════╪═══════════════╪═══════════════╪═══════════════╪════════╡
└──────────────┴──────┴───────────────┴───────────────┴───────────────┴────────┘


In [ ]:
sql2 = f'''
SELECT
    C.EndDate                AS report_date,
    C.InfoPublDate           AS date,    
    B.SecuCode               AS tick,
    C.ROETTM                 AS roe,  
    C.OperatingRevenuePSTTM  AS revenue,     
    C.OperCashFlowPSTTM      AS ocf                    
FROM (
    SELECT EndDate, InfoPublDate, CompanyCode, ROETTM, OperatingRevenuePSTTM, OperCashFlowPSTTM FROM LC_MainIndexNew
    UNION ALL
    SELECT EndDate, InfoPublDate, CompanyCode, ROETTM, OperatingRevenuePSTTM, OperCashFlowPSTTM FROM LC_STIBMainIndex
) C 
LEFT JOIN SecuMain B 
ON B.CompanyCode = C.CompanyCode
WHERE B.SecuMarket IN (83, 90)
  AND B.SecuCategory = 1         
  AND C.InfoPublDate <= '{end_dt}'        
'''

df2 = pl.from_pandas(pd.read_sql(sql2, JY_CONN))

df2 = (
    df2
    .with_columns(
        pl.col("date").dt.cast_time_unit("us")          # 时间精度对齐 df1
    )
    # 关键：同一(tick, pub_date)只保留最新财报（end_date 最大）
    .sort(["tick", "date", "report_date"])
    .unique(subset=["tick", "date"], keep="last")  
    .select(["date", "tick", "roe", "revenue", "ocf"])
    .sort(["tick", "date"])
)
df2

date,tick,roe,revenue,ocf
datetime[μs],str,f64,f64,f64
1992-03-08 00:00:00,"""000001""",174.4708,3.729,null
1992-08-11 00:00:00,"""000001""",null,2.4847,null
1993-02-27 00:00:00,"""000001""",null,3.5302,null
1993-08-07 00:00:00,"""000001""",null,1.9923,null
1994-04-16 00:00:00,"""000001""",22.6891,2.9953,null
…,…,…,…,…
2023-12-31 00:00:00,"""X26016""",14.8796,5.1438,0.854
2024-12-31 00:00:00,"""X26016""",null,5.9499,1.9415
2023-03-03 00:00:00,"""X26020""",25.7371,5.647,1.4444


In [ ]:
df2 = df2.with_columns(
    pl.col("date").dt.cast_time_unit("us")  # 转成和 df1 一样的微秒精度
)

all_dates = pl.concat([df1.select("date"), df2.select("date")]).unique()
all_ticks = pl.concat([df1.select("tick"), df2.select("tick")]).unique()
axis = all_dates.join(all_ticks, how="cross").unique()

df_full = axis.sort(['tick','date']).join(df1, on=["date", "tick"], how="left").join(df2, on=["date", "tick"], how="left")

fill_cols = ["roe", "revenue", "ocf"]  # 按你实际列名修改

df_final = (
    df_full
    .sort("tick", "date")  # 必须排序！ffill 依赖顺序
    .with_columns(
        pl.col(fill_cols).forward_fill().over("tick")
    )
)

ticks = np.load('/data/xujiayi/end2end/axis/ticks.npy', allow_pickle=True).tolist()
dts = np.load('/data/xujiayi/end2end/axis/dates.npy', allow_pickle=True).tolist()
df_final = df_final.filter(pl.col('date').is_in(dts)).filter(pl.col('tick').is_in(ticks)).unique().sort(['tick','date'])
df_final

date,tick,pe,pb,debt_ratio,is_dup,roe,revenue,ocf
datetime[μs],str,"decimal[38,4]","decimal[38,4]","decimal[38,4]",bool,f64,f64,f64
2008-01-02 00:00:00,"""000001""",34.6663,9.4762,0.8074,false,27.6584,4.7923,12.381
2008-01-03 00:00:00,"""000001""",34.0913,9.3191,0.8099,false,27.6584,4.7923,12.381
2008-01-04 00:00:00,"""000001""",35.0223,9.5736,0.8058,false,27.6584,4.7923,12.381
2008-01-07 00:00:00,"""000001""",35.7525,9.7732,0.8025,false,27.6584,4.7923,12.381
2008-01-08 00:00:00,"""000001""",40.5168,11.0755,0.7819,false,27.6584,4.7923,12.381
…,…,…,…,…,…,…,…,…
2025-12-25 00:00:00,"""688981""",204.6940,6.5134,0.1056,false,3.182,8.1787,2.8354
2025-12-26 00:00:00,"""688981""",203.0642,6.4615,0.1063,false,3.182,8.1787,2.8354
2025-12-29 00:00:00,"""688981""",203.7294,6.4827,0.1060,false,3.182,8.1787,2.8354


In [ ]:
fields = ['pe','pb','debt_ratio','roe','revenue','ocf']
for f in fields:
    feat = df_final.pivot(index='date',columns='tick',values=f).to_pandas().set_index('date')
    print(feat)
    feat.values.astype(float).tofile(f'/data/xujiayi/end2end/PINN-MTICG/IndCapGAT/{f}.bin')

             000001   000002     000004   000005    000006    000007  \
date                                                                   
2008-01-02  34.6663  73.9513  8288.0565  91.5368   41.3162  -16.5125   
2008-01-03  34.0913  72.1779  8511.1964  94.9694   41.0375  -17.3221   
2008-01-04  35.0223  74.0780  8622.7664  93.8252   43.0344  -17.3008   
2008-01-07  35.7525  76.4595  8909.6607  96.4950   44.7992  -18.1744   
2008-01-08  40.5168  75.6994  8758.2443  93.9523   43.5298  -18.1105   
...             ...      ...        ...      ...       ...       ...   
2025-12-25   5.2028  -0.9597   -11.7720     None  -12.9375   70.6793   
2025-12-26   5.1938  -0.9556   -11.7396     None  -13.0786   69.6399   
2025-12-29   5.2028  -0.9496   -11.7289     None  -12.5781   69.4565   
2025-12-30   5.1668  -0.9256   -12.1493     None  -12.6551   71.0462   
2025-12-31   5.1352  -0.9316   -11.9445     None  -12.7706   69.9456   

              000008    000009    000010     000011  ...   6887

In [ ]:
revenue = np.memmap(f'/data/xujiayi/end2end/PINN-MTICG/IndCapGAT/revenue.bin', shape=(3400, 5422), dtype=float, mode="r")
revenue

lag_revenue = np.roll(revenue, 242, axis=0)
lag_revenue[:242] = np.nan
revenue_yoy = lag_revenue / revenue - 1
revenue_yoy

revenue_yoy.astype(float).tofile(f'/data/xujiayi/end2end/PINN-MTICG/IndCapGAT/revenue_yoy.bin')

#### Indices_Mask

In [ ]:
ticks = np.load('/data/xujiayi/end2end/axis/ticks.npy', allow_pickle=True)
dts = np.load('/data/xujiayi/end2end/axis/dates.npy', allow_pickle=True)

In [ ]:
sql = f'''
select 
    B.SecuCode as 'index',
    C.SecuCode as 'tick',
    A.EndDate as 'date',
    A.Weight as 'weight'
from LC_IndexComponentsWeight A
left join SecuMain B on A.IndexCode=B.InnerCode
left join SecuMain C on A.InnerCode=C.InnerCode
where B.SecuCode in ('000300','000905','000510','000852','932000','000985')
'''
index_compt = pl.read_database(sql, JY_CONN)
index_compt

index,tick,date,weight
str,str,datetime[μs],"decimal[38,6]"
"""000510""","""000001""",2024-09-30 00:00:00,0.524000
"""000510""","""000001""",2024-10-31 00:00:00,0.496000
"""000510""","""000001""",2024-11-29 00:00:00,0.495000
"""000510""","""000001""",2024-12-31 00:00:00,0.502000
"""000510""","""000001""",2025-01-27 00:00:00,0.507000
…,…,…,…
"""000985""","""920026""",2025-12-31 00:00:00,0.002000
"""000985""","""920026""",2026-01-30 00:00:00,0.002000
"""000985""","""920026""",2026-02-27 00:00:00,0.002000


In [ ]:
index_compt = index_compt.sort(['index','date','tick'])

hs300 = index_compt.filter(pl.col('index')=='000300').pivot(index='date',columns='tick',values='weight').to_pandas().set_index('date').reindex(columns=ticks).fillna(-999)
dates = sorted(list(set(dts).union(set(hs300.index))))
hs300 = hs300.reindex(index=dates)
hs300 = hs300.ffill().reindex(dts).replace(-999,np.nan)
hs300

zz500 = index_compt.filter(pl.col('index')=='000905').pivot(index='date',columns='tick',values='weight').to_pandas().set_index('date').reindex(columns=ticks).fillna(-999)
dates = sorted(list(set(dts).union(set(zz500.index))))
zz500 = zz500.reindex(index=dates)
zz500 = zz500.ffill().reindex(dts).replace(-999,np.nan)
zz500

a500 = index_compt.filter(pl.col('index')=='000510').pivot(index='date',columns='tick',values='weight').to_pandas().set_index('date').reindex(columns=ticks).fillna(-999)
dates = sorted(list(set(dts).union(set(a500.index))))
a500 = a500.reindex(index=dates)
a500 = a500.ffill().reindex(dts).replace(-999,np.nan)
a500

zz1000 = index_compt.filter(pl.col('index')=='000852').pivot(index='date',columns='tick',values='weight').to_pandas().set_index('date').reindex(columns=ticks).fillna(-999)
dates = sorted(list(set(dts).union(set(zz1000.index))))
zz1000 = zz1000.reindex(index=dates)
zz1000 = zz1000.ffill().reindex(dts).replace(-999,np.nan)
zz1000

zz2000 = index_compt.filter(pl.col('index')=='932000').pivot(index='date',columns='tick',values='weight').to_pandas().set_index('date').reindex(columns=ticks).fillna(-999)
dates = sorted(list(set(dts).union(set(zz2000.index))))
zz2000 = zz2000.reindex(index=dates)
zz2000 = zz2000.ffill().reindex(dts).replace(-999,np.nan)
zz2000

zzfull = index_compt.filter(pl.col('index')=='000985').pivot(index='date',columns='tick',values='weight').to_pandas().set_index('date').reindex(columns=ticks).fillna(-999)
dates = sorted(list(set(dts).union(set(zzfull.index))))
zzfull = zzfull.reindex(index=dates)
zzfull = zzfull.ffill().reindex(dts).replace(-999,np.nan)
zzfull

,000001,000002,000004,000005,000006,000007,000008,000009,000010,000011,...,688796,688798,688799,688800,688802,688805,688807,688809,688819,688981
date,,,,,,,,,,,,,,,,,,,,,
2008-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-25,0.219000,0.071000,NaN,NaN,0.020000,0.005000,0.012000,0.035000,0.004000,0.004000,...,NaN,0.013000,0.006000,0.016000,NaN,NaN,NaN,NaN,0.008000,0.439000
2025-12-26,0.219000,0.071000,NaN,NaN,0.020000,0.005000,0.012000,0.035000,0.004000,0.004000,...,NaN,0.013000,0.006000,0.016000,NaN,NaN,NaN,NaN,0.008000,0.439000
2025-12-29,0.219000,0.071000,NaN,NaN,0.020000,0.005000,0.012000,0.035000,0.004000,0.004000,...,NaN,0.013000,0.006000,0.016000,NaN,NaN,NaN,NaN,0.008000,0.439000


In [ ]:
hs300.values.astype(float).tofile('/data/xujiayi/end2end/mask/hs300_weight.bin')
zz500.values.astype(float).tofile('/data/xujiayi/end2end/mask/zz500_weight.bin')
a500.values.astype(float).tofile('/data/xujiayi/end2end/mask/a500_weight.bin')
zz1000.values.astype(float).tofile('/data/xujiayi/end2end/mask/zz1000_weight.bin')
zz2000.values.astype(float).tofile('/data/xujiayi/end2end/mask/zz2000_weight.bin')
zzfull.values.astype(float).tofile('/data/xujiayi/end2end/mask/zzfull_weight.bin')

In [ ]:
hs300 = hs300.values.astype(float)
zz500 = zz500.values.astype(float)
a500 = a500.values.astype(float)
zz1000 = zz1000.values.astype(float)
zz2000 = zz2000.values.astype(float)
zzfull = zzfull.values.astype(float)

In [ ]:
hs300_mask = ~np.isnan(hs300)
hs300_mask.astype(bool).tofile('/data/xujiayi/end2end/mask/hs300_mask.bin')
zz500_mask = ~np.isnan(zz500)
zz500_mask.astype(bool).tofile('/data/xujiayi/end2end/mask/zz500_mask.bin')
a500_mask = ~np.isnan(a500)
a500_mask.astype(bool).tofile('/data/xujiayi/end2end/mask/a500_mask.bin')
zz1000_mask = ~np.isnan(zz1000)
zz1000_mask.astype(bool).tofile('/data/xujiayi/end2end/mask/zz1000_mask.bin')
zz2000_mask = ~np.isnan(zz2000)
zz2000_mask.astype(bool).tofile('/data/xujiayi/end2end/mask/zz2000_mask.bin')
zzfull_mask = ~np.isnan(zzfull)
zzfull_mask.astype(bool).tofile('/data/xujiayi/end2end/mask/zzfull_mask.bin')

#### Patch logMV for Basics

In [ ]:
sql_mv = f'''select
                    C.SecuCode as "tick",
                    A.TradingDay as "date",
                    A.NegotiableMV as "mv"
                from QT_StockPerformance A
                left join SecuMain C
                on A.InnerCode = C.InnerCode
                where A.TradingDay between '{start_dt}' and '{end_dt}'
                    and C.SecuMarket in (83,90)
                    and C.SecuCategory=1
                union all
                select
                    C.SecuCode as "tick",
                    B.TradingDay as "date",
                    B.NegotiableMV as "mv"
                from LC_STIBPerformance B
                left join SecuMain C
                on B.InnerCode = C.InnerCode
                where B.TradingDay between '{start_dt}' and '{end_dt}'
                    and C.SecuMarket in (83,90)
                    and C.SecuCategory=1
             '''
mv = pl.read_database(sql_mv, JY_CONN).sort('tick','date')
mv = mv.pivot(index='date', columns='tick', values='mv').drop('date').to_numpy().astype(float)
mv

array([[5.88761322e+10, 1.71737203e+11, 6.79013462e+08, ...,
                   nan,            nan,            nan],
       [5.78995138e+10, 1.67618805e+11, 6.97294594e+08, ...,
                   nan,            nan,            nan],
       [5.94807054e+10, 1.72031375e+11, 7.06435160e+08, ...,
                   nan,            nan,            nan],
       ...,
       [2.24328744e+11, 4.60557342e+10, 1.37401092e+09, ...,
                   nan, 3.25459080e+10, 2.44946412e+11],
       [2.22776295e+11, 4.48897663e+10, 1.42326315e+09, ...,
        5.50402356e+09, 3.25459080e+10, 2.50505196e+11],
       [2.21417903e+11, 4.51812583e+10, 1.39926847e+09, ...,
        6.25385271e+09, 3.22931620e+10, 2.45606268e+11]],
      shape=(4376, 5436))

In [ ]:
mv.astype(float).tofile('/data/xujiayi/end2end/d_field/mv.bin')

In [ ]:
logmv = np.log(mv)
logmv

array([[24.79870162, 25.86923126, 20.33615151, ...,         nan,
                nan,         nan],
       [24.78197483, 25.84495822, 20.36271854, ...,         nan,
                nan,         nan],
       [24.80891782, 25.87094271, 20.37574198, ...,         nan,
                nan,         nan],
       ...,
       [26.13637842, 24.55311811, 21.04099998, ...,         nan,
        24.20591748, 26.2243053 ],
       [26.12943395, 24.52747568, 21.07621806, ..., 22.42874522,
        24.20591748, 26.2467455 ],
       [26.12331772, 24.5339482 , 21.05921542, ..., 22.55646354,
        24.19812134, 26.22699555]], shape=(4376, 5436))

In [ ]:
logmv.astype(float).tofile('/data/xujiayi/end2end/d_field/logmv.bin')

In [ ]:
close = np.memmap('/data/xujiayi/end2end/d_field/close.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
nan_mask = np.isnan(close)
nan_mask

array([[False, False, False, ...,  True,  True,  True],
       [False, False, False, ...,  True,  True,  True],
       [False, False, False, ...,  True,  True,  True],
       ...,
       [False, False, False, ...,  True, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]],
      shape=(4376, 5436))